In [2]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import yfinance as yf


ticker = "JPM"
start_date = "2023-01-01"
end_date = "2025-01-01"

spot_price = ?
strike_price = ?

stock_price = yf.get_data(...)
risk_free_rate = yf.get_data(...)





SyntaxError: unterminated string literal (detected at line 8) (1908373099.py, line 8)

In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm
import matplotlib.pyplot as plt

# Define the ticker and the start_date
ticker = "JPM"
start_date = "2023-01-01"
end_date = "2025-01-01"

# Fetch the stock data (Adjusted Close price)
stock_data = yf.download(ticker, start=start_date, end=end_date)

# Get the stock's adjusted closing prices
stock_prices = stock_data['Close']

# Calculate daily returns (log returns)
stock_returns = np.log(stock_prices / stock_prices.shift(1))

# Example strike price and spot price (You can customize these)
strike_price = 150  # Example strike
spot_price = stock_prices.loc[start_date]  # Spot price at start_date

# Risk-free rate (for simplicity, using a fixed rate)
risk_free_rate = 0.03  # 3% annual risk-free rate (for example)

# Black-Scholes formula components (for a European call option):
def black_scholes(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + (sigma ** 2) / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price, d1, d2

# Theta calculation
def calculate_theta(S, K, T, r, sigma, d1, d2):
    theta = -S * sigma * norm.pdf(d1) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    return theta

# Delta calculation
def calculate_delta(d1):
    return norm.cdf(d1)

# Gamma calculation
def calculate_gamma(S, sigma, T, d1):
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

# Vega calculation
def calculate_vega(S, T, d1, sigma):
    return S * np.sqrt(T) * norm.pdf(d1)

# Function to calculate historical volatility over a rolling window
def calculate_volatility(stock_returns, current_day, window=90):
    start_idx = current_day - pd.Timedelta(days=window)
    historical_returns = stock_returns.loc[start_idx:current_day]
    volatility = historical_returns.std() * np.sqrt(252)  # Annualizing the volatility
    return volatility

# Set up a list to store the option prices and Greeks
option_prices = []
deltas = []
gammas = []
vegas = []
thetas = []

# Loop through each day in the stock data and calculate the option price
for current_day in stock_prices.index:
    if current_day >= pd.to_datetime(start_date) + pd.Timedelta(days=90):  # Make sure we have 90 days data
        # Calculate volatility for the past 90 days up to current_day
        volatility_90d = calculate_volatility(stock_returns, current_day, window=90)

        # Calculate time to maturity (T) in years (using 90 days for simplicity)
        T = 90 / 365  # 90 days to maturity

        # Get the spot price for the current day
        spot_price = stock_prices.loc[current_day]

        # Calculate the option price and Greeks using Black-Scholes
        call_price, d1, d2 = black_scholes(spot_price, strike_price, T, risk_free_rate, volatility_90d)

        # Calculate theta (time decay)
        theta = calculate_theta(spot_price, strike_price, T, risk_free_rate, volatility_90d, d1, d2)

        # Calculate delta
        delta = calculate_delta(d1)

        # Calculate gamma
        gamma = calculate_gamma(spot_price, volatility_90d, T, d1)

        # Calculate vega
        vega = calculate_vega(spot_price, T, d1, volatility_90d)

        # Option price adjusted by time decay (simplified approach)
        adjusted_call_price = call_price - theta * 1  # Subtracting theta * 1 day for simplicity

        # Append the values to their respective lists
        option_prices.append((current_day, adjusted_call_price))
        deltas.append((current_day, delta))
        gammas.append((current_day, gamma))
        vegas.append((current_day, vega))
        thetas.append((current_day, theta))

# Convert the lists to DataFrames
option_prices_df = pd.DataFrame(option_prices, columns=["Date", "Call Option Price"])
deltas_df = pd.DataFrame(deltas, columns=["Date", "Delta"])
gammas_df = pd.DataFrame(gammas, columns=["Date", "Gamma"])
vegas_df = pd.DataFrame(vegas, columns=["Date", "Vega"])
thetas_df = pd.DataFrame(thetas, columns=["Date", "Theta"])

# Set the Date columns as index
option_prices_df.set_index("Date", inplace=True)
deltas_df.set_index("Date", inplace=True)
gammas_df.set_index("Date", inplace=True)
vegas_df.set_index("Date", inplace=True)
thetas_df.set_index("Date", inplace=True)

# Display the option prices and Greeks time series
fig, ax = plt.subplots(2, 1, figsize=(10, 8))

# Plot the option prices
option_prices_df.plot(ax=ax[0], title="Simulated Call Option Prices Over Time (with Time Decay)", legend=False)
ax[0].set_ylabel("Option Price")

# Plot the Greeks
deltas_df.plot(ax=ax[1], title="Delta Over Time", legend=False)
gammas_df.plot(ax=ax[1], legend=False)
vegas_df.plot(ax=ax[1], legend=False)
thetas_df.plot(ax=ax[1], legend=False)
ax[1].set_ylabel("Greeks")

plt.show()


[*********************100%***********************]  1 of 1 completed


KeyError: '2023-01-01'